In [2]:
import pandas as pd
from datasets import load_dataset

def process_sportssett(dataset):
    print("Loading dataset...")
    
    csv_data = []

    print(f"Processing {len(dataset)} games...")
    
    for row in dataset:
        game_id = row.get("sportsett_id")
        
        # Keep the first summmary if "target" is not available
        if "target" in row and row["target"]:
            summary = row["target"]
        elif "summaries" in row and isinstance(row["summaries"], list) and len(row["summaries"]) > 0:
            summary = row["summaries"][0]
        else:
            summary = "No summary available"
            
        teams = row.get("teams", {})
        
        # Loop through both 'home' and 'vis' (visitor) teams
        for team_type in ["home", "vis"]:
            team_info = teams.get(team_type, {})
            team_name = team_info.get("name", "Unknown Team")
            box_score = team_info.get("box_score", [])
            
            # Loop through every player in the box score
            for player in box_score:
                player_name = player.get("name", "Unknown Player")
                
                # Extract performance metrics
                pts = player.get("PTS", "0")    # Points
                reb = player.get("TREB", "0")   # Total Rebounds
                ast = player.get("AST", "0")    # Assists
                stl = player.get("STL", "0")    # Steals
                mins = player.get("MIN", "0")   # Minutes Played
                
                # Append the player's row to our data list
                csv_data.append({
                    "sportsett_id": game_id,
                    "team": team_name,
                    "player": player_name,
                    "minutes_played": mins,
                    "points": pts,
                    "rebounds": reb,
                    "assists": ast,
                    "steals": stl,
                    "summary": summary
                })

    # Convert the list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(csv_data)
    
    # Export the DataFrame to a CSV file
    output_filename = "sportssett_players_summary.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8')
    
    print(f"Successfully processed data!")
    print(f"Created {len(df)} player rows across {len(dataset)} games.")
    print(f"Saved to: {output_filename}")

if __name__ == "__main__":
    dataset = datasets.load_dataset("parquet", data_files={"train": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/train/*.parquet", "validation": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/validation/*.parquet", "test": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/test/*.parquet"})
    process_sportssett(dataset['test'])

Loading dataset...
Processing 1230 games...
Successfully processed data!
Created 30773 player rows across 1230 games.
Saved to: sportssett_players_summary.csv


In [ ]:
import pandas as pd
import re

def filter_mentioned_players(raw_summaries):
    def is_mentioned(row):
        player_name = str(row['player'])
        summary = str(row['summary'])
        
        # 1. Check if their exact full name is in the summary
        if player_name in summary:
            return True
            
        # 2. Check last name only (if the player has more than one name part)
        name_parts = player_name.split(' ', 1)
        if len(name_parts) > 1:
            last_name = name_parts[-1]
            
            if re.search(r'\b' + re.escape(last_name) + r'\b', summary):
                return True
                
        return False

    # Apply the filter row by row
    # df.apply returns a boolean True/False for each row, filtering the dataframe
    filtered_df = df[df.apply(is_mentioned, axis=1)]
    
    # Save the cleaned data to a new CSV
    filtered_df.to_csv(output_csv_path, index=False, encoding='utf-8')
    
    print("-" * 30)
    print(f"Original player rows : {len(df)}")
    print(f"Filtered player rows : {len(filtered_df)}")
    print(f"Saved to             : {output_csv_path}")

if __name__ == "__main__":
    # Replace these with your actual file names
    INPUT_FILE = "sportssett_players_summary.csv"
    OUTPUT_FILE = "sportssett_filtered_players.csv"
    
    filter_mentioned_players(INPUT_FILE, OUTPUT_FILE)